In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType
from pyspark.sql.functions import from_json, col, from_unixtime, to_timestamp

## Read from bronze

In [0]:
df_bronze_kline = spark.readStream.table("workspace.bronze.kline")

## Transform data

In [0]:
kline_schema = StructType([
    StructField("event_time", LongType(), True),
    StructField("symbol", StringType(), True),
    StructField("start_time", LongType(), True),
    StructField("open", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("is_closed", BooleanType(), True)
])

df_silver_kline = df_bronze_kline \
    .withColumn("parsed_data", from_json(col("raw_payload"), kline_schema)) \
    .select("parsed_data.*", "ingested_at") \
    .withColumn("kline_timestamp", to_timestamp(from_unixtime(col("start_time") / 1000))) \
    .drop("start_time", "event_time") \
    .dropna()

## WRITE TO SILVER

In [0]:

silver_kline_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/03_silver/kline_1m"

query_silver_kline = (df_silver_kline.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_kline_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.silver.kline")
)

query_silver_kline.awaitTermination()
print("Hoàn tất chuyển đổi dữ liệu Nến sang Tầng Silver: workspace.silver.kline")